# Purpose:
- Calculate coding score and save

In [8]:
import sys
sys.path.append('/root/capsule/code/')
import os
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import json

from comb.behavior_ophys_dataset import BehaviorOphysDataset, BehaviorMultiplaneOphysDataset
from comb.behavior_session_dataset import BehaviorSessionDataset
from lamf_analysis.code_ocean import capsule_bod_utils as cbu
from lamf_analysis.code_ocean import capsule_data_utils as cdu

from DesignMatrix import DesignMatrix
import load_data
import design_matrix_tools as dmtools
import kernel_tools as ktools
import glm_fit_tools as gft
import coding_score as cs
import capsule_utils

# notebook dev
%load_ext autoreload
%autoreload 2
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
data_type = 'events'
version = 3
glm_path = Path('/root/capsule/scratch/glm/')

In [12]:
mouse_ids = [717824, 721291, 736963, 739564, 747107, 755252]
version = 3
data_type = 'events'

for mouse_id in mouse_ids:
    capsule_utils.process_mouse_coding_score(mouse_id, version, data_type, glm_path)


721291_2024-05-18 not processed yet
736963_2024-07-09 not processed yet
736963_2024-07-10 not processed yet
736963_2024-07-11 not processed yet
736963_2024-07-12 not processed yet
736963_2024-07-15 not processed yet
736963_2024-07-16 not processed yet
736963_2024-07-18 not processed yet
736963_2024-07-19 not processed yet
736963_2024-07-22 not processed yet
736963_2024-07-23 not processed yet
736963_2024-08-12 not processed yet
736963_2024-08-13 not processed yet
736963_2024-08-23 not processed yet
736963_2024-08-26 not processed yet
736963_2024-08-27 not processed yet
736963_2024-08-29 not processed yet


In [3]:
mouse_id = 717824
processed_date_after = None
processed_date_before = None
if mouse_id == 747107:
    processed_date_after = '2025-02-01' # inclusive
elif mouse_id == 755252:
    processed_date_after = '2025-02-07' # inclusive
elif mouse_id == 736963:
    processed_date_before = '2024-12-31' # inclusive

success, mouse_session_df = cdu.get_mouse_session_df(mouse_id,
                                                    processed_date_after=processed_date_after,
                                                    processed_date_before=processed_date_before)
if success == False:
    # either one of two reasons - duplicated raw data and missing pupil data
    # If duplicated raw data, then fail
    # If missing pupil data, then remove them
    if len(np.where(mouse_session_df.num_raw_data_asset_ids.values != 1)[0]) > 0:
        raise ValueError('Multiple raw data asset ids found for a single processed data asset id')
    else:
        missing_pupil_data = np.where(mouse_session_df.pupil_data_asset_id.values == 0)[0]
        if len(missing_pupil_data) > 0:
            print(f'Missing pupil data found for {mouse_session_df.iloc[missing_pupil_data].raw_data_date.values}')
            mouse_session_df = mouse_session_df.drop(missing_pupil_data)
assert cdu.attach_mouse_data_assets(mouse_session_df)
session_info_df = cdu.get_session_info(mouse_id)

# if natural_images_only:
session_info_df = session_info_df[session_info_df.stimulus.str.startswith('images_')]
# Another temporary filter - remove extinction sessions for now
# Due to stratification issues (with hits) - need to solve this first
session_info_df = session_info_df[~session_info_df.session_type.str.contains('OPHYS_6_')]


Not authorized to access data asset ID obtained from DocDB: subject_id=717824, id_='955c02f4-cde5-4ff8-aeaf-7ae7acb4e397'
/lamf-analysis/src/lamf_analysis/code_ocean/capsule_data_utils.py:128: UserWarning: No matching pupil data asset found for ['2024-03-20' '2024-04-30']
  warnings.warn(f'No matching pupil data asset found for {mouse_session_df[mouse_session_df["pupil_data_asset_id"] == 0].raw_data_date.values}')


Missing pupil data found for ['2024-03-20' '2024-04-30']
asset_id: 3cd9af29-f1f3-4a95-a854-0dbcfff8aca1 - mount_state: unchanged
asset_id: 1213101a-d3f0-4ee6-beed-c19d6ef2b57e - mount_state: unchanged
asset_id: 73a52715-c4a3-49c2-a5c2-d4464b3ebbe0 - mount_state: unchanged
asset_id: 5844ac91-df6a-4db8-ba62-e330858ec7b4 - mount_state: unchanged
asset_id: d07df54c-dc25-48cb-abf8-663a8ebcc86d - mount_state: unchanged
asset_id: f21853b7-21ff-4111-ae5c-5c08142175d3 - mount_state: unchanged
asset_id: e6607cfb-8c47-4df0-a23a-d58b72a6eacf - mount_state: unchanged
asset_id: 84341f60-2e16-4ba6-8c3d-50a797109fbc - mount_state: unchanged
asset_id: f292048b-f5a5-44e1-8a11-699760830888 - mount_state: unchanged


In [10]:
si = 0
session_row = session_info_df.iloc[si]
session_key = session_row.name
save_dir = glm_path / f'{session_key}_glm_v{version:02}'
glm_fn = save_dir / f'glm_results_v{version:02}_{session_key}_{data_type}.npy'

glm_results = np.load(glm_fn, allow_pickle=True).item()

X, activity_trace, activity_trace_info, run_params, unstd_features, use_indices = \
    gft.load_data(session_key, data_type, version, glm_path)

# trim X and activity_trace based on the shift
X_trim = X[use_indices, :]
activity_trace_trim = activity_trace[use_indices, :]
activity_trace_trim_filtered = gft.filter_activity_trace_matrix(activity_trace_trim)

adj_var_explained_session_model, adj_var_explained_session_model_full_mask = \
    cs.generate_session_model_adjusted_variance_explained(
        glm_results, run_params, X_trim, activity_trace_trim_filtered)
coding_score = cs.calculate_coding_score(adj_var_explained_session_model,
                                adj_var_explained_session_model_full_mask,
                                run_params)

adj_var_explained_session_model = adj_var_explained_session_model.expand_dims(
    "metric").assign_coords(metric=["adj_ve_model"])
adj_var_explained_session_model_full_mask = \
    adj_var_explained_session_model_full_mask.expand_dims(
        "metric").assign_coords(metric=["adj_ve_full"])
coding_score = coding_score.expand_dims("metric").assign_coords(
                                metric=["coding_score"])

cs_metrics = xr.concat([adj_var_explained_session_model,
                        adj_var_explained_session_model_full_mask,
                        coding_score],
                        dim="metric")
cs_metrics.name = 'coding_score_metrics_from_session_model'

save_fn = save_dir / f'coding_score_v{version:02}_{session_key}_{data_type}.nc'
cs_metrics.to_netcdf(save_fn)